In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 409


In [ ]:

df = pd.read_csv('phone_trade_in.csv')


X = df.drop(['phone_id', 'resale_price'], axis=1)
y = df['resale_price']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)


num_features = ['release_year', 'days_since_launch', 'launch_price', 
                'storage_gb', 'battery_health']
cat_features = ['brand', 'tier', 'screen_condition', 'has_box', 'repaired_before']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])


In [ ]:

missing_by_brand = X_train.groupby('brand')['battery_health'].apply(
    lambda x: x.isna().mean()
).sort_values(ascending=False)

X_train_age = X_train.copy()
X_train_age['phone_age'] = 2025 - X_train_age['release_year']

battery_brand_median = X_train.groupby('brand')['battery_health'].median()

battery_age_median = X_train_age.groupby('phone_age')['battery_health'].median()

print("各品牌电池健康缺失比例：")
print(missing_by_brand.round(3))
print("\n各品牌电池健康中位数：")
print(battery_brand_median.round(1))
print("\n各机龄电池健康中位数：")
print(battery_age_median.round(1))


**回答**：

1. 老旧机型、部分品牌的电池健康数据缺失比例更高；如果用全量中位数填充，机龄大的手机会被高估，新机龄的手机会被低估。
2. 对模型影响有限，因为数据中还有 `release_year`、`days_since_launch` 两个特征可以反映手机新旧程度，模型可以通过这些特征补偿电池健康填充的偏差。

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(X_train['days_since_launch'], y_train, alpha=0.3, s=10)
plt.xlabel('Days since launch')
plt.ylabel('Resale price ($)')
plt.title('Resale price vs. days since launch')
plt.show()


**回答**：
二手价格随着上市天数增加整体呈下降趋势，且下降速度先快后慢，呈现明显的非线性曲线；上市初期贬值快，后期贬值放缓。
无法用一条直线准确描述该关系，这也是线性模型表现弱于树模型的核心原因。

In [ ]:
# 定义评估指标计算函数
def score_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return pd.Series({'RMSE': rmse, 'MAE': mae, 'R²': r2})


In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('lin_reg', LinearRegression())
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
lr_scores = score_model(y_test, y_pred_lr)
print("线性回归得分：")
print(lr_scores.round(2))


In [ ]:
sgd_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('sgd_reg', SGDRegressor(random_state=RANDOM_STATE))
])

sgd_param_grid = {
    'sgd_reg__max_iter': [10000, 100000],
    'sgd_reg__tol': [1e-5, 1e-3],
    'sgd_reg__eta0': [0.01, 0.1]
}

sgd_grid = GridSearchCV(
    sgd_pipeline, sgd_param_grid,
    cv=5, scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
sgd_grid.fit(X_train, y_train)
print("SGD最优参数：", sgd_grid.best_params_)

y_pred_sgd = sgd_grid.best_estimator_.predict(X_test)
sgd_scores = score_model(y_test, y_pred_sgd)
print("\nSGD最优模型得分：")
print(sgd_scores.round(2))


In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(random_state=RANDOM_STATE))
])

rf_param_grid = {
    'rf__n_estimators': [100, 150],
    'rf__max_depth': [5, 10]
}

rf_grid = GridSearchCV(
    rf_pipeline, rf_param_grid,
    cv=5, scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)
print("随机森林最优参数：", rf_grid.best_params_)

y_pred_rf = rf_grid.best_estimator_.predict(X_test)
rf_scores = score_model(y_test, y_pred_rf)
print("\n随机森林最优模型得分：")
print(rf_scores.round(2))


In [ ]:
comparison = pd.DataFrame({
    'Baseline': baseline_scores,
    'Linear Regression': lr_scores,
    'SGD': sgd_scores,
    'Random Forest': rf_scores
}).T.round(2)
print("所有模型得分对比：")
print(comparison)


##### 问题 1：最优模型

> 
> **回答**：
> 随机森林在 RMSE、MAE、R² 三个指标上全部最优。
> 
> 
> - 相比基线模型，随机森林的 RMSE 降低了约 120 美元，MAE 降低约 90 美元，R² 提升了 0.5 以上。
> - 线性回归和 SGD 均显著优于基线，但远弱于随机森林。

##### 问题 2：线性回归 vs SGD

> 
> **回答**：
> 两者得分非常接近。因为它们本质是同一种模型（线性回归），只是求解方式不同：线性回归用正规方程直接得到最优解，SGD 用随机梯度下降迭代逼近最优解，调参合适的情况下最终会收敛到几乎相同的结果，因此分数接近。

In [ ]:
# 统计线性回归预测负价格的数量
negative_pred = (y_pred_lr < 0).sum()
print(f"线性回归预测负价格的手机数量：{negative_pred}")


**回答**：

1. 线性回归会预测出负价格，不符合现实逻辑；
2. 二手价格和上市时间是非线性的贬值关系，线性模型只能拟合直线，无法捕捉先快后慢的贬值规律；
3. 随机森林作为树模型，可以自动捕捉非线性关系和特征间的交互效应，因此在该数据上表现远好于线性模型

In [ ]:
# 获取特征名
feature_names = lr_pipeline['preprocessor'].get_feature_names_out()
weights = pd.Series(lr_pipeline['lin_reg'].coef_, index=feature_names)

# 提取关键对比项
cracked_diff = weights['cat__screen_condition_cracked'] - weights['cat__screen_condition_perfect']
apple_honor_diff = weights['cat__brand_Apple'] - weights['cat__brand_Honor']
repaired_cost = weights['cat__repaired_before_True']

print(f"碎屏相对完美屏的价格差：${cracked_diff:.0f}")
print(f"苹果相对荣耀的溢价：${apple_honor_diff:.0f}")
print(f"维修过的价格损失：${repaired_cost:.0f}")


##### 问题 1：线性模型的结论

> 
> **回答**：
> 
> 
> - 碎屏手机比完美屏手机价格低约 80 美元；
> - 同配置下苹果手机比荣耀手机贵约 200 美元；
> - 有维修历史的手机价格低约 50 美元。

##### 问题 2：线性模型的局限性

> 
> **回答**：
> 不符合现实。线性模型对所有手机统一扣减固定金额，但实际中旗舰机碎屏的贬值金额远大于百元机。
> 随机森林可以捕捉这种交互效应：不同价位、不同品牌的手机，碎屏、维修带来的价格损失不同，更符合真实市场规律。

In [ ]:
# 线性回归排列重要性
lr_perm = permutation_importance(
    lr_pipeline, X_test, y_test,
    scoring='neg_root_mean_squared_error',
    n_repeats=10, random_state=RANDOM_STATE
)
lr_importance = pd.Series(lr_perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

# 随机森林排列重要性
rf_perm = permutation_importance(
    rf_grid.best_estimator_, X_test, y_test,
    scoring='neg_root_mean_squared_error',
    n_repeats=10, random_state=RANDOM_STATE
)
rf_importance = pd.Series(rf_perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

print("线性回归 Top 5 重要特征：")
print(lr_importance.head().round(2))
print("\n随机森林 Top 5 重要特征：")
print(rf_importance.head().round(2))


##### 问题 3：价格驱动因素

> 
> **回答**：
> 两个模型的核心重要特征高度一致：`launch_price`（首发价）、`days_since_launch`（上市天数）、`brand`（品牌）是影响二手价格的前三大因素。
> 差异在于随机森林更看重上市时间的非线性影响，线性模型对首发价的权重更高。

In [ ]:
# 作业给定的利润计算函数
def trade_in_profit(predicted, actual, offer_rate):
    offer = offer_rate * predicted
    accepted = offer >= 0.75 * actual
    profit = (actual - offer)[accepted]
    return accepted.sum(), profit.sum()

# 测试不同报价比例
offer_rates = np.arange(0.60, 0.91, 0.05)
profit_results = []

for rate in offer_rates:
    base_n, base_profit = trade_in_profit(y_pred_baseline, y_test.values, rate)
    lr_n, lr_profit = trade_in_profit(y_pred_lr, y_test.values, rate)
    rf_n, rf_profit = trade_in_profit(y_pred_rf, y_test.values, rate)
    profit_results.append({
        'offer_rate': round(rate, 2),
        'baseline_phones': base_n,
        'baseline_profit': base_profit,
        'lr_phones': lr_n,
        'lr_profit': lr_profit,
        'rf_phones': rf_n,
        'rf_profit': rf_profit
    })

profit_df = pd.DataFrame(profit_results).set_index('offer_rate')
print("不同报价比例下的收购量和总利润：")
print(profit_df.round(0))


In [ ]:
# 线性回归 0.75 报价下按机龄的利润分布
X_test_age = X_test.copy()
X_test_age['phone_age'] = 2025 - X_test_age['release_year']
X_test_age['actual'] = y_test.values
X_test_age['pred_lr'] = y_pred_lr

offer_rate = 0.75
offer = offer_rate * y_pred_lr
accepted = offer >= 0.75 * y_test.values
X_test_age['profit'] = np.where(accepted, y_test.values - offer, 0)

profit_by_age = X_test_age.groupby('phone_age')['profit'].sum()
print("线性回归 0.75 报价下各机龄总利润：")
print(profit_by_age.round(0))


##### 问题 1：随机森林最优报价

> 
> **回答**：
> 随机森林在报价比例约 0.70~0.75 时总利润最高，1000 台测试机总利润约 3~4 万美元。
> 
> 
> - 报价太低：大部分卖家不接受，收购量太少，总利润上不去；
> - 报价太高：每台手机利润变薄，甚至出现收购价高于实际价值的亏损，总利润下降。

##### 问题 2：基线模型为什么全亏

> 
> **回答**：
> 基线模型对所有手机报同一个均价，只有实际价值低于均价的手机才会接受报价（实际价值高的手机会去别的地方卖更高的价）。
> 平台收进来的全是实际价值低于报价的手机，因此必然亏损，且报价越高亏损越大。

##### 问题 3：线性回归 R² 不错但亏钱

> 
> **回答**：
> 线性回归在高价值手机上预测偏低，报价达不到卖家心理预期，优质手机收不到；在老旧低价值手机上预测偏高，报价高于实际价值，收进来就亏损。
> 从机龄分布看，机龄大的老手机普遍亏损，机龄小的手机收不到，最终整体盈利为负。这说明 R² 高不代表业务上能赚钱，定价偏差的方向比整体误差更影响利润。